# Temă de Programare: Evaluarea Impactului Imputării

Bun venit la tema **Evaluarea Impactului Imputarii**!

Alegerea unei strategii de imputare nu inseamna doar completarea celulelor lipsa, ci si pastrarea proprietatilor statistice pe care modelele downstream se bazeaza. Teoria introduce doua instrumente-cheie de evaluare:

- **Testul Kolmogorov–Smirnov** — verifica daca distributia imputata poate fi distinsa statistic de distributia observata. Un p-value mare ($\gg 0.05$) si o statistica KS mica ($< 0.15$) indica faptul ca imputarea a pastrat forma distributiei.
- **Norma Frobenius** pe matricea de corelatie — $\|R^{after} - R^{before}\|_F = \sqrt{\sum_{ij}(r^{after}_{ij} - r^{before}_{ij})^2}$. O valoare apropiata de 0 inseamna ca relatiile perechi au ramas intacte; peste 1.50 semnaleaza o distorsiune severa.

**Ce vei face in aceasta tema:**

* Vei stabili un model de baza antrenat pe date complete
* Vei masura cum afecteaza listwise deletion performanta modelului
* Vei compara impactul imputarii simple fata de cea avansata asupra metricilor modelului
* Vei construi un raport cuprinzator de impact pentru toate strategiile
* Vei evalua fidelitatea distributiilor folosind teste KS si norma Frobenius

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI TALE:</h4>

* Toate celulele sunt blocate cu excepția celor în care trebuie să trimiți soluțiile sau atunci când este menționat explicit că poți interacționa cu ele.

* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga sau modifica niciun cod care se află în afara acestor comentarii**.

* Poți adăuga celule noi pentru a experimenta, dar acestea vor fi omise de evaluator, deci nu te baza pe celule create nou pentru a găzdui codul soluției — folosește locurile prevăzute în acest scop.

* **Evită utilizarea variabilelor globale** cu excepția cazului în care este absolut necesar. Evaluatorul testează codul tău într-un mediu izolat.

---

## Cuprins
- [Importuri](#imports)
- [1 - Pregatirea Datelor](#1---data-preparation)
    - [1.1 - Generarea setului de date complet](#11---generating-complete-dataset)
    - [1.2 - Introducerea valorilor lipsa](#12---introducing-missing-values)
- [2 - Model de Referinta](#2---baseline-model)
    - **[Exercitiul 1 - `prepare_baseline_model`](#exercise-1---prepare_baseline_model)**
- [3 - Impactul Stergerii](#3---deletion-impact)
    - **[Exercitiul 2 - `apply_listwise_deletion`](#exercise-2---apply_listwise_deletion)**
- [4 - Impactul Imputarii Simple](#4---simple-imputation-impact)
    - **[Exercitiul 3 - `apply_simple_imputation`](#exercise-3---apply_simple_imputation)**
- [5 - Impactul Imputarii Avansate](#5---advanced-imputation-impact)
    - **[Exercitiul 4 - `apply_advanced_imputation`](#exercise-4---apply_advanced_imputation)**
- [6 - Raport Comprehensiv de Impact](#6---comprehensive-impact-report)
    - **[Exercitiul 5 - `create_impact_report`](#exercise-5---create_impact_report)**

*Nota: Exercitiile 3–5 primesc un **singur** set de date (nu split train/test), iar testele gestioneaza intern impartirea.*


<a name='imports'></a>
## Importuri

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

In [ ]:
import helper_utils
import unittests

<a name='1---data-preparation'></a>
## 1 - Pregătirea Datelor

<a name='11---generating-complete-dataset'></a>
### 1.1 - Generarea Setului de Date Complet

Generam un set de date sintetic cu relatii cunoscute intre trasaturi si tinta. Acesta serveste drept **ground truth** — baza fata de care masuram impactul diferitelor strategii de tratare a datelor lipsa.


In [ ]:
# Generate complete dataset (no missing values)
X_complete, y = helper_utils.generate_complete_dataset(n_samples=500, n_features=5, random_state=42)

print(f"Dataset shape: {X_complete.shape}")
print(f"Missing values: {X_complete.isnull().sum().sum()}")
print(f"\nFeature statistics:")
X_complete.describe().round(2)

<a name='12---introducing-missing-values'></a>
### 1.2 - Introducerea Valorilor Lipsa

Introducem lipsa de tip MCAR (rata 20%). In regim MCAR, relatia ground truth dintre trasaturi si tinta este pastrata in datele observate — riscul de bias este mic, dar pierdem totusi putere statistica.


In [ ]:
X_missing = helper_utils.introduce_missing_values(X_complete, missing_rate=0.2, random_state=42)

print("Missing values per feature:")
print(X_missing.isnull().sum())
print(f"\nTotal missing: {X_missing.isnull().sum().sum()}")
print(f"Missing %:     {X_missing.isnull().sum().sum() / X_missing.size * 100:.1f}%")

In [ ]:
helper_utils.plot_missing_pattern(X_missing)

<a name='2---baseline-model'></a>
## 2 - Modelul de Referinta

Inainte de a evalua strategiile de imputare, antrenam un model pe **date complete** (fara valori lipsa). Aceasta este limita superioara de performanta, adica cel mai bun caz la care ne putem astepta sa se apropie orice strategie de imputare.


<a name='exercise-1---prepare_baseline_model'></a>
### **Exercitiul 1 - `prepare_baseline_model`**

**Sarcina ta:**

Implementeaza `prepare_baseline_model(X_train, X_test, y_train, y_test)` astfel incat sa antreneze un `LinearRegression` pe date complete si sa returneze metricile de performanta.

**Cerinte:**
- Antreneaza un model `LinearRegression`
- Returneaza un `dict` cu cheile `'rmse'`, `'mae'`, `'r2'`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere daca te-ai blocat)</font></b></summary>

**Pasul 1:** `model = LinearRegression()` apoi `model.fit(X_train, y_train)`

**Pasul 2:** `y_pred = model.predict(X_test)`

**Pasul 3:**
- `rmse = np.sqrt(mean_squared_error(y_test, y_pred))`
- `mae  = mean_absolute_error(y_test, y_pred)`
- `r2   = r2_score(y_test, y_pred)`

</details>


In [ ]:
# GRADED FUNCTION: prepare_baseline_model
def prepare_baseline_model(X_train, X_test, y_train, y_test):
    """
    Train a baseline Linear Regression model and return performance metrics.

    Args:
        X_train (pd.DataFrame): Training features (complete, no missing values).
        X_test  (pd.DataFrame): Test features.
        y_train (pd.Series): Training target.
        y_test  (pd.Series): Test target.

    Returns:
        dict: {'rmse': float, 'mae': float, 'r2': float}
    """
    ### ÎNCEPUT DE COD AICI ###

    model  = None
    # Fit the model
    None

    y_pred = None

    rmse = None
    mae  = None
    r2   = None

    ### SFÂRȘIT DE COD AICI ###

    return {'rmse': rmse, 'mae': mae, 'r2': r2}

In [ ]:
X_train_c, X_test_c, y_train, y_test = train_test_split(
    X_complete, y, test_size=0.2, random_state=42
)

baseline = prepare_baseline_model(X_train_c, X_test_c, y_train, y_test)
print("Baseline (Complete Data):")
print(f"  RMSE: {baseline['rmse']:.4f}")
print(f"  MAE:  {baseline['mae']:.4f}")
print(f"  R2:   {baseline['r2']:.4f}")

In [ ]:
# Testează-ți codul!
unittests.exercise_1(prepare_baseline_model)

<a name='3---deletion-impact'></a>
## 3 - Impactul Stergerii

Listwise deletion este strategia de baza. Intrebarea-cheie este: cat de mult reduce performanta pierderea randurilor si daca esantionul ramas mai este reprezentativ?


<a name='exercise-2---apply_listwise_deletion'></a>
### **Exercitiul 2 - `apply_listwise_deletion`**

**Sarcina ta:**

Implementeaza `apply_listwise_deletion(X, y)` astfel incat sa elimine randurile in care lipseste orice trasatura si sa alinieze `y` in consecinta.

**Cerinte:**
- Returneaza un tuplu `(X_clean, y_clean, deleted_count)`
- `X_clean` nu trebuie sa contina valori lipsa
- `y_clean` trebuie sa aiba aceeasi lungime cu `X_clean`
- `deleted_count` este un intreg: numarul de randuri eliminate

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

**Pasul 1:** `complete_mask = X.notnull().all(axis=1)` — masca booleana pentru randurile complete

**Pasul 2:** `X_clean = X[complete_mask]`, `y_clean = y[complete_mask]`

**Pasul 3:** `deleted_count = (~complete_mask).sum()`

</details>


In [ ]:
# GRADED FUNCTION: apply_listwise_deletion
def apply_listwise_deletion(X, y):
    """
    Drop rows with any missing feature and align the target vector.

    Args:
        X (pd.DataFrame): Feature matrix with potential missing values.
        y (pd.Series): Target vector aligned with X.

    Returns:
        tuple: (X_clean, y_clean, deleted_count)
    """
    ### ÎNCEPUT DE COD AICI ###

    complete_mask  = None
    X_clean        = None
    y_clean        = None
    deleted_count  = None

    ### SFÂRȘIT DE COD AICI ###

    return X_clean, y_clean, deleted_count

In [ ]:
X_clean, y_clean, n_deleted = apply_listwise_deletion(X_missing, y)
print(f"Original rows:  {len(X_missing)}")
print(f"Rows deleted:   {n_deleted}")
print(f"Remaining rows: {len(X_clean)}")
print(f"Missing values: {X_clean.isnull().sum().sum()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_2(apply_listwise_deletion)

<a name='4---simple-imputation-impact'></a>
## 4 - Impactul Imputarii Simple

Spre deosebire de stergere, imputarea pastreaza dimensiunea esantionului, dar poate introduce bias. Imputarea cu media este cea mai comuna, dar se stie ca **subestimeaza varianta** si **distorsioneaza corelatiile**.


<a name='exercise-3---apply_simple_imputation'></a>
### **Exercitiul 3 - `apply_simple_imputation`**

**Sarcina ta:**

Implementeaza `apply_simple_imputation(X_missing, strategy='mean')` astfel incat sa antreneze un `SimpleImputer` pe datele furnizate si sa returneze atat DataFrame-ul imputat, cat si imputatorul antrenat.

**Cerinte:**
- Antreneaza un `SimpleImputer(strategy=strategy)` pe `X_missing`
- Returneaza un tuplu `(X_imputed, imputer)` unde `X_imputed` este un `pd.DataFrame` fara valori lipsa, iar `imputer` este obiectul `SimpleImputer` deja antrenat

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

```python
imputer = SimpleImputer(strategy=strategy)
imputer.fit(X_missing)
X_imputed = pd.DataFrame(
    imputer.transform(X_missing),
    columns=X_missing.columns,
    index=X_missing.index
)
return X_imputed, imputer
```

</details>


In [ ]:
# GRADED FUNCTION: apply_simple_imputation
def apply_simple_imputation(X_train, X_test, strategy='mean'):
    """
    Impute training and test sets using a simple strategy, fitting only on training data.

    Args:
        X_train (pd.DataFrame): Training features with missing values.
        X_test  (pd.DataFrame): Test features with missing values.
        strategy (str): 'mean' or 'median'.

    Returns:
        tuple: (X_train_imputed, X_test_imputed) — both pd.DataFrames.
    """
    ### ÎNCEPUT DE COD AICI ###

    # Create imputer and fit ONLY on training data
    imputer = None
    None  # fit

    # Transform both sets using training statistics
    X_train_imputed = None
    X_test_imputed  = None

    ### SFÂRȘIT DE COD AICI ###

    return X_train_imputed, X_test_imputed

In [ ]:
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_missing, y, test_size=0.2, random_state=42
)

X_tr_imp, X_te_imp = apply_simple_imputation(X_train_m, X_test_m, strategy='mean')
print(f"Training missing after mean imputation: {X_tr_imp.isnull().sum().sum()}")
print(f"Test missing after mean imputation:     {X_te_imp.isnull().sum().sum()}")

mean_model = prepare_baseline_model(X_tr_imp, X_te_imp, y_train_m, y_test_m)
print(f"\nMean imputation — RMSE: {mean_model['rmse']:.4f}, R2: {mean_model['r2']:.4f}")

In [ ]:
# Testează-ți codul!
unittests.exercise_3(apply_simple_imputation)

<a name='5---advanced-imputation-impact'></a>
## 5 - Impactul Imputarii Avansate

Metodele avansate (KNN, iterative) exploateaza structura multivariata si, in general, produc valori imputate mai apropiate de valorile reale in date de tip MAR, cu pretul unui cost computational mai ridicat.


<a name='exercise-4---apply_advanced_imputation'></a>
### **Exercitiul 4 - `apply_advanced_imputation`**

**Sarcina ta:**

Implementeaza `apply_advanced_imputation(X_missing)` astfel incat sa aplice **atat** imputarea KNN, cat si imputarea iterativa pe acelasi set de date si sa returneze ambele rezultate.

**Cerinte:**
- Aplica independent `KNNImputer(n_neighbors=5)` si `IterativeImputer(max_iter=10, random_state=42)`
- Returneaza un tuplu `(X_knn, X_iterative)` — ambele fiind `pd.DataFrame` fara valori lipsa

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

```python
knn_imputer  = KNNImputer(n_neighbors=5)
iter_imputer = IterativeImputer(max_iter=10, random_state=42)

X_knn = pd.DataFrame(knn_imputer.fit_transform(X_missing),
                     columns=X_missing.columns, index=X_missing.index)
X_iterative = pd.DataFrame(iter_imputer.fit_transform(X_missing),
                            columns=X_missing.columns, index=X_missing.index)
```

</details>


In [ ]:
# GRADED FUNCTION: apply_advanced_imputation
def apply_advanced_imputation(X_train, X_test, method='knn'):
    """
    Apply KNN or iterative imputation with proper fit/transform separation.

    Args:
        X_train (pd.DataFrame): Training features with missing values.
        X_test  (pd.DataFrame): Test features with missing values.
        method (str): 'knn' or 'iterative'.

    Returns:
        tuple: (X_train_imputed, X_test_imputed) — both pd.DataFrames.
    """
    ### ÎNCEPUT DE COD AICI ###

    if method == 'knn':
        imputer = None
    else:
        imputer = None

    None  # fit on training data only

    X_train_imputed = None
    X_test_imputed  = None

    ### SFÂRȘIT DE COD AICI ###

    return X_train_imputed, X_test_imputed

In [ ]:
X_tr_knn, X_te_knn = apply_advanced_imputation(X_train_m, X_test_m, method='knn')
knn_model = prepare_baseline_model(X_tr_knn, X_te_knn, y_train_m, y_test_m)
print(f"KNN imputation       — RMSE: {knn_model['rmse']:.4f}, R2: {knn_model['r2']:.4f}")

X_tr_iter, X_te_iter = apply_advanced_imputation(X_train_m, X_test_m, method='iterative')
iter_model = prepare_baseline_model(X_tr_iter, X_te_iter, y_train_m, y_test_m)
print(f"Iterative imputation — RMSE: {iter_model['rmse']:.4f}, R2: {iter_model['r2']:.4f}")

In [ ]:
# Testează-ți codul!
unittests.exercise_4(apply_advanced_imputation)

<a name='6---comprehensive-impact-report'></a>
## 6 - Raport Comprehensiv de Impact

Reuneste toate strategiile intr-un singur raport comparativ care afiseaza atat performanta modelului, cat si metricile de fidelitate a distributiei.


<a name='exercise-5---create_impact_report'></a>
### **Exercitiul 5 - `create_impact_report`**

**Sarcina ta:**

Implementeaza `create_impact_report(results_dict, original_data, imputed_data)` astfel incat sa compare strategiile de imputare si sa o identifice pe cea mai buna.

**Cerinte:**
- `results_dict`: un dictionar care mapeaza numele strategiei → `{'rmse': ..., 'mae': ..., 'r2': ...}`
- `original_data`: un `pd.Series` cu valorile originale (inainte de introducerea lipsei)
- `imputed_data`: un `pd.Series` cu valorile imputate pentru comparatie
- Returneaza un **dict** cu cheile:
  - `'summary_table'` — `pd.DataFrame` care rezuma toate strategiile
  - `'best_strategy'` — numele strategiei cu cel mai mic RMSE
  - `'distribution_metrics'` — dictionar cu rezultatele testului KS care compara `original_data` vs `imputed_data`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

```python
from scipy import stats

summary_table = pd.DataFrame([
    {'strategy': k, **v} for k, v in results_dict.items()
])

best_strategy = summary_table.loc[summary_table['rmse'].idxmin(), 'strategy']

ks_stat, ks_p = stats.ks_2samp(original_data.dropna(), imputed_data.dropna())
distribution_metrics = {'ks_statistic': ks_stat, 'ks_p_value': ks_p}

return {
    'summary_table': summary_table,
    'best_strategy': best_strategy,
    'distribution_metrics': distribution_metrics
}
```

</details>


In [ ]:
# GRADED FUNCTION: create_impact_report
def create_impact_report(X_complete, X_missing, y):
    """
    Build a comprehensive comparison of imputation strategies on model performance.

    Args:
        X_complete (pd.DataFrame): Complete feature matrix (no missing values).
        X_missing  (pd.DataFrame): Feature matrix with missing values.
        y (pd.Series): Target vector.

    Returns:
        pd.DataFrame: Rows are strategies; columns include 'strategy', 'rmse', 'r2', 'n_samples'.
    """
    ### ÎNCEPUT DE COD AICI ###

    rows   = []
    rs     = 42  # random_state

    # ---- Baseline (complete data) ----
    X_tr_c, X_te_c, y_tr, y_te = train_test_split(X_complete, y, test_size=0.2, random_state=rs)
    metrics = prepare_baseline_model(X_tr_c, X_te_c, y_tr, y_te)
    rows.append({'strategy': 'baseline', 'n_samples': len(X_complete),
                 'rmse': metrics['rmse'], 'r2': metrics['r2']})

    # ---- Listwise deletion ----
    X_del, y_del, _ = None  # apply_listwise_deletion
    if len(X_del) > 10:
        X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_del, y_del, test_size=0.2, random_state=rs)
        metrics = prepare_baseline_model(X_tr_d, X_te_d, y_tr_d, y_te_d)
        rows.append({'strategy': 'deletion', 'n_samples': len(X_del),
                     'rmse': metrics['rmse'], 'r2': metrics['r2']})

    # ---- Mean imputation ----
    X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X_missing, y, test_size=0.2, random_state=rs)
    for strategy in ['mean', 'median']:
        X_tr_imp, X_te_imp = None  # apply_simple_imputation
        metrics = prepare_baseline_model(X_tr_imp, X_te_imp, y_tr_m, y_te_m)
        rows.append({'strategy': strategy, 'n_samples': len(X_tr_imp) + len(X_te_imp),
                     'rmse': metrics['rmse'], 'r2': metrics['r2']})

    # ---- KNN and Iterative ----
    for method in ['knn', 'iterative']:
        X_tr_adv, X_te_adv = None  # apply_advanced_imputation
        metrics = prepare_baseline_model(X_tr_adv, X_te_adv, y_tr_m, y_te_m)
        rows.append({'strategy': method, 'n_samples': len(X_tr_adv) + len(X_te_adv),
                     'rmse': metrics['rmse'], 'r2': metrics['r2']})

    report = None  # pd.DataFrame(rows)

    ### SFÂRȘIT DE COD AICI ###

    return report

In [ ]:
report = create_impact_report(X_complete, X_missing, y)
print("Imputation Impact Report:")
report.sort_values('rmse')

In [ ]:
# Testează-ți codul!
unittests.exercise_5(create_impact_report)

In [ ]:
# Visualize the comparison
if report is not None:
    helper_utils.plot_impact_comparison(report)